# Solution Generation and Enhancement for Railroad Construction

This notebook generates comprehensive solution files with detailed metrics and information for Streamlit analysis and visualization.

## Purpose
- Load optimized solutions from Pareto front results
- Enrich solutions with detailed resource assignments and metrics
- Calculate performance indicators (distances, violations, utilization)
- Export structured solution files for dashboard analysis

## Workflow
1. **Data Loading**: Import problem instance and solution data
2. **Resource Assignment**: Map workers to tasks and calculate assignments
3. **Metrics Calculation**: Compute distances, violations, and utilization rates
4. **Solution Export**: Generate enriched JSON files for analysis

## Output Format
- Structured JSON with complete resource assignments
- Calculated metrics for machines, workers, and attachments
- Performance indicators ready for Streamlit visualization

In [ ]:
"""
Solution Generation and Enhancement Script

This script processes optimized railroad construction solutions and generates
comprehensive output files with detailed resource assignments, performance
metrics, and analysis data for Streamlit visualization.

Features:
- Multi-solution batch processing
- Detailed resource assignment mapping
- Comprehensive metrics calculation
- Distance and violation analysis
- Structured JSON export for dashboards
"""

import json

# =============================================================================
# CONFIGURATION AND INPUT PARAMETERS
# =============================================================================

# Solution IDs to process from the Pareto front
loesungs_ids = ["1_178", "14_210"]  # Selected high-quality solutions for analysis

# Process each solution individually
for loesung_id in loesungs_ids:
    # File paths for input data and output generation
    instance_file = "Instanzen/Real_Life/2_piece/Construction_RealLife_2024_7_1_2.json"        # Problem instance
    solution_file = "Solutions_RealLife/RealLife_2024_7_1_2/First_400_2/pareto_solutions_filtered.json"  # Pareto solutions
    output_file = f"generated_solution_{loesung_id}.json"  # Enhanced solution export
    #output_file = f"Solutions_RealLife/RealLife_2024_7_1_2/First_400_2/generated_solution_{loesung_id}.json"

    # =============================================================================
    # DATA LOADING AND PREPROCESSING
    # =============================================================================

    # Load problem instance data with orders, resources, and constraints
    with open(instance_file, "r") as f:
        instance_data = json.load(f)

    # Load optimized solution data from Pareto front results
    with open(solution_file, "r") as f:
        solution_data = json.load(f)

    # Create lookup dictionaries for efficient data access
    bp_dict = {bp["ID"]: bp for bp in instance_data["Bestellpositionen"]}      # Order positions lookup
    maschinen_infos = {m["ID"]: m for m in instance_data["Maschinen"]}         # Machine information lookup
    transportwege = instance_data.get("TransportwegeString", {})               # Transportation distance matrix

    # Extract specific solution data for processing
    sol_entry = solution_data[loesung_id]

    # =============================================================================
    # WORKER ASSIGNMENT MAPPING
    # =============================================================================
    
    # Build worker assignments per order position for detailed resource tracking
    bp_to_workers = {}
    if "worker_route_plan" in sol_entry:
        for worker_id, bp_ids in sol_entry["worker_route_plan"].items():
            for bp_id in bp_ids:
                bp_to_workers.setdefault(bp_id, []).append(int(worker_id))

    # =============================================================================
    # SOLUTION ENRICHMENT FUNCTIONS
    # =============================================================================

    def enrich_bp(bp_id, maschinentyp="N/A", zugewiesene_arbeiter=None):
        """
        Enrich order position with detailed information for analysis.
        
        Combines basic order position data with resource assignments and
        scheduling information to create comprehensive task descriptions.
        
        Args:
            bp_id: Order position identifier
            maschinentyp: Machine type assignment (optional)
            zugewiesene_arbeiter: List of assigned worker IDs
            
        Returns:
            dict: Enriched order position with all relevant information
        """
        bp = bp_dict.get(bp_id)
        if not bp:
            return None
        
        return {
            "ID": bp["ID"],                                                         # Unique identifier
            "Start": bp["Start"],                                                   # Start time
            "Ende": bp["Ende"],                                                     # End time
            "Dauer": bp["Dauer"],                                                   # Duration
            "Auftragsnummer": bp["Auftragsnummer"],                                 # Order number
            "MaschinenTyp": maschinentyp,                                           # Machine type requirement
            "AnbaugeraeteTypen": bp.get("AnbaugeraeteTypen", []),                   # Required attachment types
            "ArbeiterQualifikationen": bp.get("ArbeiterQualifikationen", []),       # Required worker qualifications
            "ZugewieseneArbeiter": zugewiesene_arbeiter if zugewiesene_arbeiter else [],  # Assigned workers
            "zugewieseneMaschine": None,                                            # Assigned machine (placeholder)
            "Typ": "schicht"                                                        # Task type identifier
        }

    # =============================================================================
    # OUTPUT STRUCTURE INITIALIZATION
    # =============================================================================

    # Initialize comprehensive output structure for enhanced solution data
    output_json = {
        "Version": "2025_01",                                                       # Solution format version
        "RechenzeitInSekunden": 0,                                                  # Computation time placeholder
        "BerechnetAuftragBearbeitet": {},                                          # Order completion status
        "MaschinenLoesung": {"Maschinenzuweisung": {}},                            # Machine assignments
        "Arbeiterloesung": {"Arbeiterzuweisung": {}},                              # Worker assignments
        "AnbaugeraeteLoesung": {"Anbaugeraetzuweisung": {}}                        # Attachment assignments
    }

    # Track order completion status for analysis
    auftrag_status = {}

    # =============================================================================
    # ROUTE PLAN PROCESSING FUNCTIONS
    # =============================================================================

    def process_maschinenroute(routeplan, out_dict, prefix=""):
        """
        Process machine route plans with worker assignment integration.
        
        Processes machine scheduling data and integrates worker assignments
        to create comprehensive resource allocation information.
        
        Args:
            routeplan: Machine route planning data
            out_dict: Output dictionary for processed assignments
            prefix: Resource identifier prefix
        """
        for res_id, bp_ids in routeplan.items():
            key = f"{prefix}{res_id}"
            out_dict[key] = []
            
            for bp_id in bp_ids:
                # Get assigned workers for this order position
                zugewiesene_arbeiter = bp_to_workers.get(bp_id, [])
                enriched = enrich_bp(bp_id, zugewiesene_arbeiter=zugewiesene_arbeiter)
                
                if enriched:
                    out_dict[key].append(enriched)
                    auftrags_id = int(enriched["Auftragsnummer"])
                    auftrag_status[f"Auftrag {auftrags_id}"] = True  # Mark order as processed

    def process_routeplan(routeplan, out_dict, prefix=""):
        """
        Generic route plan processing for workers and attachments.
        
        Processes resource scheduling data and creates structured
        assignments for analysis and visualization.
        
        Args:
            routeplan: Resource route planning data
            out_dict: Output dictionary for processed assignments
            prefix: Resource identifier prefix
        """
        for res_id, bp_ids in routeplan.items():
            key = f"{prefix}{res_id}"
            out_dict[key] = []
            
            for bp_id in bp_ids:
                enriched = enrich_bp(bp_id)
                if enriched:
                    out_dict[key].append(enriched)
                    auftrags_id = int(enriched["Auftragsnummer"])
                    auftrag_status[f"Auftrag {auftrags_id}"] = True  # Mark order as processed

    # =============================================================================
    # ROUTE PLAN EXECUTION
    # =============================================================================

    # Process machine assignments with integrated worker data
    if "machine_route_plan" in sol_entry:
        process_maschinenroute(sol_entry["machine_route_plan"], 
                             output_json["MaschinenLoesung"]["Maschinenzuweisung"], 
                             prefix="M_")

    # Process worker assignments and scheduling
    if "worker_route_plan" in sol_entry:
        process_routeplan(sol_entry["worker_route_plan"], 
                         output_json["Arbeiterloesung"]["Arbeiterzuweisung"], 
                         prefix="Arbeiter_")

    # Process attachment assignments and allocation
    if "attachment_route_plan" in sol_entry:
        process_routeplan(sol_entry["attachment_route_plan"], 
                         output_json["AnbaugeraeteLoesung"]["Anbaugeraetzuweisung"], 
                         prefix="ABG_")

    # =============================================================================
    # ORDER COMPLETION STATUS TRACKING
    # =============================================================================

    # Mark unprocessed orders as incomplete for comprehensive status tracking
    alle_auftragsnummern = set(bp["Auftragsnummer"] for bp in instance_data["Bestellpositionen"])
    for auftragsnummer in alle_auftragsnummern:
        key = f"Auftrag {int(auftragsnummer)}"
        if key not in auftrag_status:
            auftrag_status[key] = False  # Mark as not processed
    
    # Sort order status by order number for organized output
    output_json["BerechnetAuftragBearbeitet"] = dict(sorted(auftrag_status.items(), 
                                                           key=lambda x: int(x[0].split()[1])))

    # =============================================================================
    # MACHINE METRICS CALCULATION
    # =============================================================================

    # Calculate comprehensive machine performance metrics
    verletzungen_pro_maschine = {}      # Driver constraint violations per machine
    maschine_genutzt = {}               # Machine utilization status
    kilometer_pro_maschine = {}         # Transportation distances per machine

    for maschine_id, einsatzliste in output_json["MaschinenLoesung"]["Maschinenzuweisung"].items():
        # Extract machine index and load machine-specific data
        maschine_idx = int(maschine_id.split("_")[-1])
        maschine_data = maschinen_infos.get(maschine_idx, {})
        stammfahrer = set(maschine_data.get("StammfahrerStrings", []))  # Authorized drivers

        # Initialize metrics tracking
        verletzungen = 0        # Driver constraint violations
        km_summe = 0.0          # Total transportation distance
        last_auftragsnr = None  # Previous order for distance calculation

        for einsatz in einsatzliste:
            curr_auftragsnr = einsatz["Auftragsnummer"]
            zugewiesene_arbeiter = einsatz.get("ZugewieseneArbeiter", [])

            # Check driver authorization constraint
            has_stammfahrer = any(str(aid) in stammfahrer for aid in zugewiesene_arbeiter)
            if not has_stammfahrer:
                verletzungen += 1  # Count constraint violation

            # Calculate transportation distance between consecutive orders
            if last_auftragsnr is not None:
                str_from = str(last_auftragsnr)
                str_to = str(curr_auftragsnr)
                km = transportwege.get(str_from, {}).get(str_to, 0.0)
                km_summe += km

            last_auftragsnr = curr_auftragsnr

        # Store calculated metrics
        verletzungen_pro_maschine[maschine_id] = verletzungen
        maschine_genutzt[maschine_id] = len(einsatzliste) > 0
        kilometer_pro_maschine[maschine_id] = km_summe

    # Add comprehensive machine metrics to output
    output_json["MaschinenLoesung"].update({
        "BerechneteStammfahrerVerletzungenProMaschine": verletzungen_pro_maschine,  # Driver violations per machine
        "BerechnetMaschineGenutzt": maschine_genutzt,                              # Machine utilization flags
        "BerechneteKilometer": kilometer_pro_maschine,                             # Distance per machine
        "BerechneteKilometerGesamt": sum(kilometer_pro_maschine.values()),         # Total machine distances
        "AnzahlGenutzterMaschinen": sum(maschine_genutzt.values()),                # Total machines used
        "AnzahlStammfahrerVerletzungen": sum(verletzungen_pro_maschine.values())   # Total driver violations
    })

    # =============================================================================
    # WORKER METRICS CALCULATION
    # =============================================================================

    # Load worker commuting distance data
    arbeitswege = instance_data.get("ArbeitswegeString", {})

    # Calculate worker-specific performance metrics
    arbeitsweg_pro_arbeiter = {}        # Commuting distances per worker
    arbeiter_genutzt = {}               # Worker utilization status

    for arbeiter_key, einsatzliste in output_json["Arbeiterloesung"]["Arbeiterzuweisung"].items():
        # Extract worker ID from key format "Arbeiter_5"
        arbeiter_id = arbeiter_key.replace("Arbeiter_", "")
        
        km_summe = 0.0                  # Total commuting distance
        genutzt = len(einsatzliste) > 0 # Worker utilization flag

        for einsatz in einsatzliste:
            auftragsnr = einsatz["Auftragsnummer"]
            # Calculate commuting distance from worker to order location
            km = arbeitswege.get(arbeiter_id, {}).get(str(auftragsnr), 0.0)
            km_summe += km

        # Double distance for round-trip commuting
        arbeitsweg_pro_arbeiter[arbeiter_key] = 2 * km_summe
        arbeiter_genutzt[arbeiter_key] = genutzt

    # Calculate total worker utilization
    anzahl_genutzte_arbeiter = sum(arbeiter_genutzt.values())

    # Add comprehensive worker metrics to output
    output_json["Arbeiterloesung"].update({
        "BerechneteKilometer": arbeitsweg_pro_arbeiter,                            # Commuting distances per worker
        "BerechneteKilometerGesamt": sum(arbeitsweg_pro_arbeiter.values()),        # Total worker commuting distance
        "BerechnetArbeiterGenutzt": arbeiter_genutzt,                              # Worker utilization flags
        "AnzahlGenutzterArbeiter": anzahl_genutzte_arbeiter                        # Total workers used
    })

    # =============================================================================
    # ATTACHMENT METRICS CALCULATION
    # =============================================================================

    # Calculate attachment-specific performance metrics
    anbau_km = {}                       # Transportation distances per attachment
    anbau_genutzt = {}                  # Attachment utilization status

    for geraet_id, einsatzliste in output_json["AnbaugeraeteLoesung"]["Anbaugeraetzuweisung"].items():
        km_summe = 0.0                  # Total transportation distance
        genutzt = len(einsatzliste) > 0 # Attachment utilization flag
        last_auftragsnr = None          # Previous order for distance calculation

        for einsatz in einsatzliste:
            curr_auftragsnr = einsatz["Auftragsnummer"]

            # Calculate transportation distance between consecutive orders
            if last_auftragsnr is not None:
                str_from = str(last_auftragsnr)
                str_to = str(curr_auftragsnr)
                km = transportwege.get(str_from, {}).get(str_to, 0.0)
                km_summe += km

            last_auftragsnr = curr_auftragsnr

        # Store calculated metrics
        anbau_km[geraet_id] = km_summe
        anbau_genutzt[geraet_id] = genutzt

    # Calculate total attachment utilization
    anzahl_genutzt = sum(anbau_genutzt.values())

    # Add comprehensive attachment metrics to output
    output_json["AnbaugeraeteLoesung"].update({
        "BerechneteKilometer": anbau_km,                                           # Transportation distances per attachment
        "BerechneteKilometerGesamt": sum(anbau_km.values()),                       # Total attachment transportation distance
        "BerechnetAnbaugeraetGenutzt": anbau_genutzt,                              # Attachment utilization flags
        "AnzahlGenutzterAnbaugeraete": anzahl_genutzt                              # Total attachments used
    })

    # =============================================================================
    # SOLUTION EXPORT
    # =============================================================================

    # Export enhanced solution with comprehensive metrics to JSON file
    with open(output_file, "w") as f:
        json.dump(output_json, f, indent=4)

    print(f"✅ Enhanced solution file generated: {output_file}")
    print(f"📊 Orders processed: {sum(auftrag_status.values())}/{len(auftrag_status)}")
    print(f"🚛 Machines used: {sum(maschine_genutzt.values())}")
    print(f"👷 Workers used: {anzahl_genutzte_arbeiter}")
    print(f"🔧 Attachments used: {anzahl_genutzt}")
    print(f"📍 Total distance: {sum(kilometer_pro_maschine.values()):.2f} km")
    print("-" * 50)

✅ Datei geschrieben: generated_solution_1_178.json
✅ Datei geschrieben: generated_solution_14_210.json
